# Request Validation with Pydantic

---

In this notebook, we dive deeper into **Pydantic**, the data validation library that powers FastAPI's input/output handling. Good validation is what separates a prototype API from a production-grade one.

We will cover:
- Advanced field validators (ranges, patterns, custom logic)
- Model configuration with `model_config`
- Handling batch predictions (multiple samples in one request)
- Custom error messages
- The full schemas used in our `app/` project

---

## 1. Why Validation Matters for ML APIs

Your scikit-learn Pipeline will happily accept any array of 4 numbers, even nonsensical ones. It won't crash if you pass `sepal_length = -999` or `petal_width = 1000000`. It will just silently return a garbage prediction.

**Validation** is your first line of defense. It ensures that the data reaching your model is at least *plausible* before inference happens.

---

## 2. Field Validators

Pydantic's `Field()` function lets you add constraints beyond just types:

In [ ]:
from pydantic import BaseModel, Field

class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, le=10, description="Sepal length in cm")
    sepal_width: float = Field(..., gt=0, le=6, description="Sepal width in cm")
    petal_length: float = Field(..., gt=0, le=8, description="Petal length in cm")
    petal_width: float = Field(..., gt=0, le=4, description="Petal width in cm")

### Available Numeric Constraints

| Constraint | Meaning | Example |
| :--- | :--- | :--- |
| `gt` | Greater than | `gt=0` → must be > 0 |
| `ge` | Greater than or equal to | `ge=0` → must be ≥ 0 |
| `lt` | Less than | `lt=100` → must be < 100 |
| `le` | Less than or equal to | `le=10` → must be ≤ 10 |

### Why Set Upper Bounds?
The Iris dataset's features have known ranges. A `sepal_length` of 100 cm is clearly an error (that would be a meter-long sepal). By setting `le=10`, we can catch obvious mistakes. The exact bounds depend on your domain knowledge.

---

## 3. Custom Validators

For validation logic that goes beyond simple numeric ranges, Pydantic provides the `@field_validator` decorator.

In [ ]:
from pydantic import BaseModel, Field, field_validator

class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, le=10, description="Sepal length in cm")
    sepal_width: float = Field(..., gt=0, le=6, description="Sepal width in cm")
    petal_length: float = Field(..., gt=0, le=8, description="Petal length in cm")
    petal_width: float = Field(..., gt=0, le=4, description="Petal width in cm")

    @field_validator("petal_length")
    @classmethod
    def petal_length_must_be_less_than_sepal(cls, v, info):
        sepal_length = info.data.get("sepal_length")
        if sepal_length is not None and v > sepal_length:
            raise ValueError(
                f"Petal length ({v}) should not exceed sepal length ({sepal_length})"
            )
        return v

This is a cross-field validator: it checks that `petal_length` doesn't exceed `sepal_length`, which is biologically implausible for iris flowers.

> 💡 **When to use custom validators:** Only add them when you have clear domain knowledge. Don't over-validate; the model itself should handle edge cases within plausible ranges.

---

## 4. Model Configuration and Examples

Pydantic models can include example data that appears in the Swagger UI. This makes your API self-documenting:

In [ ]:
class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, le=10, description="Sepal length in cm")
    sepal_width: float = Field(..., gt=0, le=6, description="Sepal width in cm")
    petal_length: float = Field(..., gt=0, le=8, description="Petal length in cm")
    petal_width: float = Field(..., gt=0, le=4, description="Petal width in cm")

    model_config = {
        "json_schema_extra" : {
            "examples": [
                {
                    "sepal_length": 5.1,
                    "sepal_width": 3.5,
                    "petal_length": 1.4,
                    "petal_width": 0.2
                },
            ] 
        }
    }

Now when a user opens Swagger UI, the example values are pre-filled. They can click "Execute" immediately without guessing what values to enter.

---

## 5. Batch Predictions

Predicting one sample at a time is fine for interactive users, but real-world clients often want to send multiple samples in one request. This is more efficient because it avoids the overhead of one HTTP round-trip per prediction

In [ ]:
@app.post("/predict/batch", response_model=list[IrisPrediction])
def predict_batch(samples: list[IrisFeatures]):
    # Convert all samples to a single NumPy array
    X = np.array([
        [s.sepal_length, s.sepal_width, s.petal_length, s.petal_width] for s in samples
        ])
    
    pipeline = ml_model["pipeline"]
    predictions = pipeline.predict(X)
    probabilities = pipeline.predict_proba(X)
    
    return [
        IrisPrediction(
            prediction=TARGET_NAMES[int(pred)],
            prediction_id = int(pred),
            probabilities={
                name: round(float(prob), 4) for name, prob in zip(TARGET_NAMES, probs)
            }
        )
        for pred, probs in zip(predictions, probabilities)
    ]

The client sends a JSON array:
```json
[
    {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
    {"sepal_length": 6.7, "sepal_width": 3.0, "petal_length": 5.2, "petal_width": 2.3}
]
```

And receives a JSON array of predictions. Each sample is individually validated by Pydantic.

---

## 6. Error Responses in Practice

When validation fails, FastAPI automatically returns a structured error. For example, sending `{"sepal_length": -1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}` would return:

```json
{
    "detail": [
        {
            "type": "greater_than",
            "loc": ["body", "sepal_length"],
            "msg": "Input should be greater than 0",
            "input": -1,
            "ctx": {"gt": 0}
        }
    ]
}
```

This tells the client exactly:
- **Where** the error is (`sepal_length` in the request body)
- **What** went wrong (value should be greater than 0)
- **What** was sent (-1)

You get this for free, no error handling code needed!

---

## 7. Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **`Field()` constraints** | Use `gt`, `ge`, `lt`, `le` to enforce numeric ranges. |
| **`@field_validator`** | Custom validation logic for cross-field checks or domain rules. |
| **`model_config` / examples** | Pre-fill Swagger UI with example data for self-documenting APIs. |
| **Batch endpoint** | Accept `list[IrisFeatures]` for efficient multi-sample predictions. |
| **Automatic errors** | Pydantic returns structured 422 errors with field-level detail. No manual error handling needed. |

---

**Next:** Explore the complete working application in the `app/` subfolder, then move on to [Containerization](../03_containerization/) to package it all with Docker.